# Autoencoders — Implementations

Each autoencoder again on tensors, with the hand-derived backward passes replaced by autograd, plus `torch.nn` lanes where the module has an exact counterpart. Every random draw — init weights, corruption noise, the VAE's eps — comes from NumPy generators shared across lanes, so the equivalence deltas measure arithmetic, not luck.

## 17_linear_autoencoder

The narrowest network that is secretly PCA.

### torch

The same class on tensors, with the four hand-derived gradients replaced by one `loss.backward()`. Init weights are drawn with NumPy's generator and `as_tensor`'d, so both lanes start from identical numbers. **What torch adds:** autograd — delete the backward half of `fit` and nothing else changes.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Draw W_e and W_d with NumPy's generator, then as_tensor — torch's RNG never matches.
# 2. requires_grad_(True) on all four tensors; loss.backward() replaces the backward block.
# 3. Set p.grad = None before each backward, and update inside torch.no_grad().
# 4. float64 end to end — agreeing with the NumPy lane is the whole point.


def xavier_init(d_in, d_out, rng):
    """The NumPy lane's Xavier draw, kept in NumPy so both lanes share the numbers."""
    return rng.normal(0, np.sqrt(2.0 / (d_in + d_out)), size=(d_in, d_out))


class LinearAutoencoder:
    """Linear autoencoder on tensors: identical init draws, autograd for the backward."""

    def __init__(self, d_input, d_latent, random_state=42):
        rng_init = np.random.default_rng(random_state)
        self.W_e = torch.as_tensor(xavier_init(d_input, d_latent, rng_init))
        self.b_e = torch.zeros(d_latent, dtype=torch.float64)
        self.W_d = torch.as_tensor(xavier_init(d_latent, d_input, rng_init))
        self.b_d = torch.zeros(d_input, dtype=torch.float64)

    def encode(self, X):
        if not torch.is_tensor(X):
            X = torch.as_tensor(np.asarray(X, dtype=float))
        return X @ self.W_e + self.b_e

    def decode(self, Z):
        if not torch.is_tensor(Z):
            Z = torch.as_tensor(np.asarray(Z, dtype=float))
        return Z @ self.W_d + self.b_d

    def forward(self, X):
        Z = self.encode(X)
        X_hat = self.decode(Z)
        return X_hat, Z

    def fit(self, X, lr=0.001, n_steps=2000, verbose=True):
        """Full-batch gradient descent; one backward() replaces four derived gradients."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        params = [self.W_e, self.b_e, self.W_d, self.b_d]
        for p in params:
            p.requires_grad_(True)
        history = []
        for step in range(n_steps):
            X_hat = self.decode(self.encode(Xt))
            loss = torch.mean((X_hat - Xt) ** 2)
            history.append(float(loss))
            for p in params:
                p.grad = None
            loss.backward()
            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad
            if verbose and (step + 1) % 500 == 0:
                print(f'  step {step + 1}/{n_steps}, loss = {history[-1]:.6f}')
        for p in params:
            p.requires_grad_(False)
        return history


In [ ]:
# exports: final_loss, recon_head, z_head
_rng_eq = np.random.default_rng(1700)
_t_eq = _rng_eq.uniform(0, 2 * np.pi, 60)
_r_eq = 1.0 + 0.3 * _rng_eq.normal(size=60)
_Zlat_eq = np.column_stack([_r_eq * np.cos(_t_eq), _r_eq * np.sin(_t_eq)])
_A_eq = _rng_eq.normal(size=(2, 6))
_X_eq = _Zlat_eq @ _A_eq + 0.1 * _rng_eq.normal(size=(60, 6))
_X_eq = _X_eq - _X_eq.mean(axis=0)

_ae_eq = LinearAutoencoder(6, 2, random_state=3)
_hist_eq = _ae_eq.fit(_X_eq, lr=0.05, n_steps=400, verbose=False)
_Xh_eq, _Zc_eq = _ae_eq.forward(_X_eq)
final_loss = _hist_eq[-1]
recon_head = _Xh_eq[:5].detach().numpy()
z_head = _Zc_eq[:5].detach().numpy()
print(f"final reconstruction MSE: {final_loss:.6f}")


In [ ]:
# Real properties of a linear autoencoder, not of this fixture.
assert final_loss < _hist_eq[0] * 0.5, "training should at least halve the loss"
_U_eq, _S_eq, _Vt_eq = np.linalg.svd(_X_eq, full_matrices=False)
_mse_pca_eq = float(np.mean((_X_eq - _X_eq @ _Vt_eq[:2].T @ _Vt_eq[:2]) ** 2))
assert final_loss >= _mse_pca_eq - 1e-9, "no linear AE beats the rank-2 PCA floor (Eckart-Young)"
_Xh_all_eq = _ae_eq.forward(_X_eq)[0].detach().numpy()
assert np.linalg.matrix_rank(_Xh_all_eq - _Xh_all_eq.mean(0), tol=1e-8) <= 2, \
    "a k=2 bottleneck confines reconstructions to a rank-2 subspace"


### library

`nn.Linear` encoder and decoder under `optim.SGD` and `nn.MSELoss`, weights copied from the same NumPy draw — transposed, because `nn.Linear` stores `(out, in)`. **What the library adds:** module and optimiser plumbing; the update it performs is exactly the one the notebook derived.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# hints:
# 1. nn.Linear stores weight as (out, in) and computes x @ W.T — copy your draw transposed.
# 2. Zero the biases after copying weights; nn.Linear initialises them randomly.
# 3. nn.MSELoss() means the mean over every element — the notebook's mse_loss exactly.
# 4. SGD without momentum is exactly param -= lr * grad — trajectories match stepwise.


def xavier_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / (d_in + d_out)), size=(d_in, d_out))


class LinearAutoencoder:
    """nn.Linear encoder/decoder trained by optim.SGD, seeded from the same NumPy draw."""

    def __init__(self, d_input, d_latent, random_state=42):
        rng_init = np.random.default_rng(random_state)
        W_e = xavier_init(d_input, d_latent, rng_init)
        W_d = xavier_init(d_latent, d_input, rng_init)
        self.encoder = nn.Linear(d_input, d_latent).double()
        self.decoder = nn.Linear(d_latent, d_input).double()
        with torch.no_grad():
            self.encoder.weight.copy_(torch.as_tensor(W_e.T))
            self.encoder.bias.zero_()
            self.decoder.weight.copy_(torch.as_tensor(W_d.T))
            self.decoder.bias.zero_()

    def encode(self, X):
        if not torch.is_tensor(X):
            X = torch.as_tensor(np.asarray(X, dtype=float))
        return self.encoder(X)

    def decode(self, Z):
        if not torch.is_tensor(Z):
            Z = torch.as_tensor(np.asarray(Z, dtype=float))
        return self.decoder(Z)

    def forward(self, X):
        Z = self.encode(X)
        return self.decode(Z), Z

    def fit(self, X, lr=0.001, n_steps=2000, verbose=True):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        opt = torch.optim.SGD([*self.encoder.parameters(),
                               *self.decoder.parameters()], lr=lr)
        loss_fn = nn.MSELoss()
        history = []
        for step in range(n_steps):
            opt.zero_grad()
            loss = loss_fn(self.decoder(self.encoder(Xt)), Xt)
            history.append(float(loss))
            loss.backward()
            opt.step()
            if verbose and (step + 1) % 500 == 0:
                print(f'  step {step + 1}/{n_steps}, loss = {history[-1]:.6f}')
        return history


In [ ]:
# exports: final_loss, recon_head, z_head
_rng_eq = np.random.default_rng(1700)
_t_eq = _rng_eq.uniform(0, 2 * np.pi, 60)
_r_eq = 1.0 + 0.3 * _rng_eq.normal(size=60)
_Zlat_eq = np.column_stack([_r_eq * np.cos(_t_eq), _r_eq * np.sin(_t_eq)])
_A_eq = _rng_eq.normal(size=(2, 6))
_X_eq = _Zlat_eq @ _A_eq + 0.1 * _rng_eq.normal(size=(60, 6))
_X_eq = _X_eq - _X_eq.mean(axis=0)

_ae_eq = LinearAutoencoder(6, 2, random_state=3)
_hist_eq = _ae_eq.fit(_X_eq, lr=0.05, n_steps=400, verbose=False)
_Xh_eq, _Zc_eq = _ae_eq.forward(_X_eq)
final_loss = _hist_eq[-1]
recon_head = _Xh_eq[:5].detach().numpy()
z_head = _Zc_eq[:5].detach().numpy()
print(f"final reconstruction MSE: {final_loss:.6f}")


In [ ]:
assert tuple(_ae_eq.encoder.weight.shape) == (2, 6), "nn.Linear keeps weight as (out, in)"
assert final_loss < _hist_eq[0] * 0.5, "SGD training reduced the loss"
_U_eq, _S_eq, _Vt_eq = np.linalg.svd(_X_eq, full_matrices=False)
_mse_pca_eq = float(np.mean((_X_eq - _X_eq @ _Vt_eq[:2].T @ _Vt_eq[:2]) ** 2))
assert final_loss >= _mse_pca_eq - 1e-9, "PCA is the floor for any linear autoencoder"


## 17_nonlinear_autoencoder

ReLU layers bend the subspace into a manifold.

### torch

The 6→12→2→12→6 net with the whole backprop block — eight gradients, two ReLU masks, cached activations — collapsed into `loss.backward()`. **What torch adds:** the tape; the NumPy lane's `self._H1`/`self._A1` bookkeeping is what autograd automates.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Same generator order as NumPy: W1 (He), W2 (Xavier), W3 (He), W4 (Xavier).
# 2. torch.relu in forward; autograd applies relu_deriv's mask for you in backward.
# 3. No cached activations needed — the graph remembers _H1 and _A1 for you.
# 4. Fixed step counts, no early exit: every lane must train exactly as long.


def he_init(d_in, d_out, rng):
    """He init for the ReLU layers, drawn with NumPy so both lanes share it."""
    return rng.normal(0, np.sqrt(2.0 / d_in), size=(d_in, d_out))


def xavier_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / (d_in + d_out)), size=(d_in, d_out))


class NonlinearAutoencoder:
    """d -> h (ReLU) -> k -> h (ReLU) -> d on tensors; autograd does the backward."""

    def __init__(self, d_input, d_hidden, d_latent, random_state=42):
        r = np.random.default_rng(random_state)
        self.W1 = torch.as_tensor(he_init(d_input, d_hidden, r))
        self.b1 = torch.zeros(d_hidden, dtype=torch.float64)
        self.W2 = torch.as_tensor(xavier_init(d_hidden, d_latent, r))
        self.b2 = torch.zeros(d_latent, dtype=torch.float64)
        self.W3 = torch.as_tensor(he_init(d_latent, d_hidden, r))
        self.b3 = torch.zeros(d_hidden, dtype=torch.float64)
        self.W4 = torch.as_tensor(xavier_init(d_hidden, d_input, r))
        self.b4 = torch.zeros(d_input, dtype=torch.float64)

    def encode(self, X):
        if not torch.is_tensor(X):
            X = torch.as_tensor(np.asarray(X, dtype=float))
        A1 = torch.relu(X @ self.W1 + self.b1)
        return A1 @ self.W2 + self.b2

    def decode(self, Z):
        if not torch.is_tensor(Z):
            Z = torch.as_tensor(np.asarray(Z, dtype=float))
        A3 = torch.relu(Z @ self.W3 + self.b3)
        return A3 @ self.W4 + self.b4

    def forward(self, X):
        Z = self.encode(X)
        return self.decode(Z), Z

    def fit(self, X, lr=0.001, n_steps=3000, verbose=True):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        params = [self.W1, self.b1, self.W2, self.b2,
                  self.W3, self.b3, self.W4, self.b4]
        for p in params:
            p.requires_grad_(True)
        history = []
        for step in range(n_steps):
            X_hat, _ = self.forward(Xt)
            loss = torch.mean((X_hat - Xt) ** 2)
            history.append(float(loss))
            for p in params:
                p.grad = None
            loss.backward()
            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad
            if verbose and (step + 1) % 1000 == 0:
                print(f'  step {step + 1}/{n_steps}, loss = {history[-1]:.6f}')
        for p in params:
            p.requires_grad_(False)
        return history


In [ ]:
# exports: final_loss, recon_head, z_head
_rng_eq = np.random.default_rng(1700)
_t_eq = _rng_eq.uniform(0, 2 * np.pi, 60)
_r_eq = 1.0 + 0.3 * _rng_eq.normal(size=60)
_Zlat_eq = np.column_stack([_r_eq * np.cos(_t_eq), _r_eq * np.sin(_t_eq)])
_A_eq = _rng_eq.normal(size=(2, 6))
_X_eq = _Zlat_eq @ _A_eq + 0.1 * _rng_eq.normal(size=(60, 6))
_X_eq = _X_eq - _X_eq.mean(axis=0)

_ae_eq = NonlinearAutoencoder(6, d_hidden=12, d_latent=2, random_state=5)
_hist_eq = _ae_eq.fit(_X_eq, lr=0.01, n_steps=400, verbose=False)
_Xh_eq, _Zc_eq = _ae_eq.forward(_X_eq)
final_loss = _hist_eq[-1]
recon_head = _Xh_eq[:5].detach().numpy()
z_head = _Zc_eq[:5].detach().numpy()
print(f"final reconstruction MSE: {final_loss:.6f}")


In [ ]:
assert final_loss < _hist_eq[0] * 0.5, "training reduced reconstruction error"
# random_state pins the init draws: a fresh model reproduces the first-step loss.
_ae2_eq = NonlinearAutoencoder(6, d_hidden=12, d_latent=2, random_state=5)
_Xh0_eq, _ = _ae2_eq.forward(_X_eq)
_loss0_eq = float(torch.mean((_Xh0_eq - torch.as_tensor(_X_eq)) ** 2))
assert abs(_loss0_eq - _hist_eq[0]) < 1e-12, "random_state makes the init reproducible"
assert np.asarray(z_head).shape == (5, 2), "the bottleneck really is 2-dimensional"


### library

The same net as two `nn.Sequential` halves with weights copied from the identical NumPy draws. **What the library adds:** composability — swapping depth or activation is a one-line change, and `optim.SGD` performs the very update written by hand.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# hints:
# 1. nn.Sequential(Linear, ReLU, Linear) is one half; index into it to reach each layer.
# 2. Copy each weight transposed — nn.Linear is (out, in) — and zero every bias.
# 3. One optimiser over encoder + decoder parameters; opt.step() is the whole update.
# 4. nn.MSELoss defaults to the mean over all elements, matching the notebook's loss.


def he_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / d_in), size=(d_in, d_out))


def xavier_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / (d_in + d_out)), size=(d_in, d_out))


class NonlinearAutoencoder:
    """The same 6->12->2->12->6 net as two nn.Sequential halves, same init draws."""

    def __init__(self, d_input, d_hidden, d_latent, random_state=42):
        r = np.random.default_rng(random_state)
        W1 = he_init(d_input, d_hidden, r)
        W2 = xavier_init(d_hidden, d_latent, r)
        W3 = he_init(d_latent, d_hidden, r)
        W4 = xavier_init(d_hidden, d_input, r)
        self.encoder = nn.Sequential(nn.Linear(d_input, d_hidden), nn.ReLU(),
                                     nn.Linear(d_hidden, d_latent)).double()
        self.decoder = nn.Sequential(nn.Linear(d_latent, d_hidden), nn.ReLU(),
                                     nn.Linear(d_hidden, d_input)).double()
        with torch.no_grad():
            self.encoder[0].weight.copy_(torch.as_tensor(W1.T))
            self.encoder[2].weight.copy_(torch.as_tensor(W2.T))
            self.decoder[0].weight.copy_(torch.as_tensor(W3.T))
            self.decoder[2].weight.copy_(torch.as_tensor(W4.T))
            for layer in (self.encoder[0], self.encoder[2],
                          self.decoder[0], self.decoder[2]):
                layer.bias.zero_()

    def encode(self, X):
        if not torch.is_tensor(X):
            X = torch.as_tensor(np.asarray(X, dtype=float))
        return self.encoder(X)

    def decode(self, Z):
        if not torch.is_tensor(Z):
            Z = torch.as_tensor(np.asarray(Z, dtype=float))
        return self.decoder(Z)

    def forward(self, X):
        Z = self.encode(X)
        return self.decode(Z), Z

    def fit(self, X, lr=0.001, n_steps=3000, verbose=True):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        opt = torch.optim.SGD([*self.encoder.parameters(),
                               *self.decoder.parameters()], lr=lr)
        loss_fn = nn.MSELoss()
        history = []
        for step in range(n_steps):
            opt.zero_grad()
            loss = loss_fn(self.decoder(self.encoder(Xt)), Xt)
            history.append(float(loss))
            loss.backward()
            opt.step()
            if verbose and (step + 1) % 1000 == 0:
                print(f'  step {step + 1}/{n_steps}, loss = {history[-1]:.6f}')
        return history


In [ ]:
# exports: final_loss, recon_head, z_head
_rng_eq = np.random.default_rng(1700)
_t_eq = _rng_eq.uniform(0, 2 * np.pi, 60)
_r_eq = 1.0 + 0.3 * _rng_eq.normal(size=60)
_Zlat_eq = np.column_stack([_r_eq * np.cos(_t_eq), _r_eq * np.sin(_t_eq)])
_A_eq = _rng_eq.normal(size=(2, 6))
_X_eq = _Zlat_eq @ _A_eq + 0.1 * _rng_eq.normal(size=(60, 6))
_X_eq = _X_eq - _X_eq.mean(axis=0)

_ae_eq = NonlinearAutoencoder(6, d_hidden=12, d_latent=2, random_state=5)
_hist_eq = _ae_eq.fit(_X_eq, lr=0.01, n_steps=400, verbose=False)
_Xh_eq, _Zc_eq = _ae_eq.forward(_X_eq)
final_loss = _hist_eq[-1]
recon_head = _Xh_eq[:5].detach().numpy()
z_head = _Zc_eq[:5].detach().numpy()
print(f"final reconstruction MSE: {final_loss:.6f}")


In [ ]:
assert tuple(_ae_eq.encoder[0].weight.shape) == (12, 6), "first Linear is (out, in) = (12, 6)"
assert isinstance(_ae_eq.encoder[1], nn.ReLU), "the nonlinearity sits between the linear maps"
assert final_loss < _hist_eq[0] * 0.5, "optim.SGD training reduced the loss"


## 17_denoising_autoencoder

Reconstruct the clean signal from a corrupted view. *No library lane:* `torch.nn` ships no denoising estimator — corruption is a training-loop choice, so a library lane would only repeat the torch lane with new plumbing.

### torch

Corrupt with NumPy noise from `default_rng(142)` (the NumPy lane's `SEED + 100`), forward the noisy input, take the loss against the clean target. **What torch adds:** the encoder gradient through `X_noisy` comes for free — the NumPy lane must remember to use the noisy input in `dW1`, a classic silent bug.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Corruption noise comes from np.random.default_rng(142) — the NumPy lane's SEED + 100.
# 2. Draw the noise with NumPy each step and as_tensor it: same draws, same order.
# 3. Forward the noisy input, but compute the loss against the clean X.
# 4. Autograd already uses X_noisy in the encoder gradient — by hand you must remember to.


def he_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / d_in), size=(d_in, d_out))


def xavier_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / (d_in + d_out)), size=(d_in, d_out))


class DenoisingAutoencoder:
    """The NumPy lane subclasses NonlinearAutoencoder; lanes are standalone, so the
    architecture is written out again — everything new lives in fit's first two lines."""

    def __init__(self, d_input, d_hidden, d_latent, random_state=42):
        r = np.random.default_rng(random_state)
        self.W1 = torch.as_tensor(he_init(d_input, d_hidden, r))
        self.b1 = torch.zeros(d_hidden, dtype=torch.float64)
        self.W2 = torch.as_tensor(xavier_init(d_hidden, d_latent, r))
        self.b2 = torch.zeros(d_latent, dtype=torch.float64)
        self.W3 = torch.as_tensor(he_init(d_latent, d_hidden, r))
        self.b3 = torch.zeros(d_hidden, dtype=torch.float64)
        self.W4 = torch.as_tensor(xavier_init(d_hidden, d_input, r))
        self.b4 = torch.zeros(d_input, dtype=torch.float64)

    def encode(self, X):
        if not torch.is_tensor(X):
            X = torch.as_tensor(np.asarray(X, dtype=float))
        A1 = torch.relu(X @ self.W1 + self.b1)
        return A1 @ self.W2 + self.b2

    def decode(self, Z):
        if not torch.is_tensor(Z):
            Z = torch.as_tensor(np.asarray(Z, dtype=float))
        A3 = torch.relu(Z @ self.W3 + self.b3)
        return A3 @ self.W4 + self.b4

    def forward(self, X):
        Z = self.encode(X)
        return self.decode(Z), Z

    def fit(self, X, noise_std=0.5, lr=0.003, n_steps=3000, verbose=True):
        """Corrupt the input each step; reconstruct the clean target."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        n, d = Xt.shape
        rng_noise = np.random.default_rng(142)  # SEED + 100, as in the NumPy lane
        params = [self.W1, self.b1, self.W2, self.b2,
                  self.W3, self.b3, self.W4, self.b4]
        for p in params:
            p.requires_grad_(True)
        history = []
        for step in range(n_steps):
            noise = torch.as_tensor(rng_noise.normal(0, noise_std, size=(n, d)))
            X_hat, _ = self.forward(Xt + noise)
            loss = torch.mean((X_hat - Xt) ** 2)
            history.append(float(loss))
            for p in params:
                p.grad = None
            loss.backward()
            with torch.no_grad():
                for p in params:
                    p -= lr * p.grad
            if verbose and (step + 1) % 1000 == 0:
                print(f'  step {step + 1}/{n_steps}, loss = {history[-1]:.6f}')
        for p in params:
            p.requires_grad_(False)
        return history


In [ ]:
# exports: final_loss, denoised_head
_rng_eq = np.random.default_rng(1700)
_t_eq = _rng_eq.uniform(0, 2 * np.pi, 60)
_r_eq = 1.0 + 0.3 * _rng_eq.normal(size=60)
_Zlat_eq = np.column_stack([_r_eq * np.cos(_t_eq), _r_eq * np.sin(_t_eq)])
_A_eq = _rng_eq.normal(size=(2, 6))
_X_eq = _Zlat_eq @ _A_eq + 0.1 * _rng_eq.normal(size=(60, 6))
_X_eq = _X_eq - _X_eq.mean(axis=0)

_dae_eq = DenoisingAutoencoder(6, d_hidden=12, d_latent=2, random_state=7)
_hist_eq = _dae_eq.fit(_X_eq, noise_std=0.5, lr=0.02, n_steps=1500, verbose=False)
_rng_test_eq = np.random.default_rng(555)
_Xn_eq = _X_eq + _rng_test_eq.normal(0, 0.5, size=_X_eq.shape)
_Xd_eq, _Zd_eq = _dae_eq.forward(_Xn_eq)
final_loss = _hist_eq[-1]
denoised_head = _Xd_eq[:5].detach().numpy()
print(f"final training loss (clean target): {final_loss:.6f}")


In [ ]:
_mse_noisy_eq = float(np.mean((_Xn_eq - _X_eq) ** 2))
_mse_den_eq = float(torch.mean((_Xd_eq - torch.as_tensor(_X_eq)) ** 2))
assert _mse_den_eq < _mse_noisy_eq, "denoising must beat leaving the noise in place"
assert final_loss < _hist_eq[0], "training reduced the clean-target loss"
assert final_loss < _mse_noisy_eq, "the model reconstructs clean X below the corruption level"


## 17_vae

A latent distribution instead of a latent point. *No library lane:* `torch.nn` has no VAE estimator — the model is exactly this handful of layers plus a loss, which the torch lane already shows.

### torch

Same encoder/decoder, same NumPy eps in `reparameterize`, and one `total_loss.backward()` in place of the notebook's ten hand-derived gradients — including the per-gradient norm clip at 5.0, mirrored exactly. **What torch adds:** the chain rule through the reparameterisation trick, which is the derivation's hardest part.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Init order matters: W1, W_mu, W_lv, W3, W4 from one default_rng(random_state).
# 2. eps comes from the NumPy generator too — reparameterise with as_tensor'd noise.
# 3. One total_loss = recon + beta*KL, one backward: autograd covers the reparam chain rule.
# 4. Clip each gradient's norm at 5.0 separately, exactly like the NumPy loop.
# 5. np.random.default_rng(242) inside fit is SEED + 200 — the NumPy lane's training noise.


def he_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / d_in), size=(d_in, d_out))


def xavier_init(d_in, d_out, rng):
    return rng.normal(0, np.sqrt(2.0 / (d_in + d_out)), size=(d_in, d_out))


class VAE:
    """VAE on tensors: same init draws, same NumPy noise, autograd for the ELBO."""

    def __init__(self, d_input, d_hidden, d_latent, random_state=42):
        r = np.random.default_rng(random_state)
        self.d_latent = d_latent
        self.W1 = torch.as_tensor(he_init(d_input, d_hidden, r))
        self.b1 = torch.zeros(d_hidden, dtype=torch.float64)
        self.W_mu = torch.as_tensor(xavier_init(d_hidden, d_latent, r))
        self.b_mu = torch.zeros(d_latent, dtype=torch.float64)
        self.W_lv = torch.as_tensor(xavier_init(d_hidden, d_latent, r))
        self.b_lv = torch.zeros(d_latent, dtype=torch.float64)
        self.W3 = torch.as_tensor(he_init(d_latent, d_hidden, r))
        self.b3 = torch.zeros(d_hidden, dtype=torch.float64)
        self.W4 = torch.as_tensor(xavier_init(d_hidden, d_input, r))
        self.b4 = torch.zeros(d_input, dtype=torch.float64)

    def encode(self, X):
        """Return mu, log_var."""
        if not torch.is_tensor(X):
            X = torch.as_tensor(np.asarray(X, dtype=float))
        A1 = torch.relu(X @ self.W1 + self.b1)
        mu = A1 @ self.W_mu + self.b_mu
        log_var = A1 @ self.W_lv + self.b_lv
        return mu, log_var

    def reparameterize(self, mu, log_var, rng_sample):
        """z = mu + sigma * eps; eps is drawn with NumPy so every lane sees the same z."""
        std = torch.exp(0.5 * log_var)
        eps = torch.as_tensor(rng_sample.normal(size=tuple(mu.shape)))
        return mu + std * eps

    def decode(self, Z):
        if not torch.is_tensor(Z):
            Z = torch.as_tensor(np.asarray(Z, dtype=float))
        A3 = torch.relu(Z @ self.W3 + self.b3)
        return A3 @ self.W4 + self.b4

    @staticmethod
    def kl_divergence(mu, log_var):
        """KL(q(z|x) || N(0,I)) = 0.5 * sum(mu^2 + var - log_var - 1)."""
        return 0.5 * torch.sum(mu ** 2 + torch.exp(log_var) - log_var - 1, dim=1).mean()

    def forward(self, X, rng_sample):
        mu, log_var = self.encode(X)
        Z = self.reparameterize(mu, log_var, rng_sample)
        X_hat = self.decode(Z)
        return X_hat, mu, log_var, Z

    def fit(self, X, lr=0.001, n_steps=5000, beta=1.0, verbose=True):
        """Train on recon + beta*KL; autograd differentiates through the reparam trick."""
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        rng_train = np.random.default_rng(242)  # SEED + 200, as in the NumPy lane
        params = [self.W1, self.b1, self.W_mu, self.b_mu, self.W_lv, self.b_lv,
                  self.W3, self.b3, self.W4, self.b4]
        for p in params:
            p.requires_grad_(True)
        history = {'loss': [], 'recon': [], 'kl': []}
        for step in range(n_steps):
            X_hat, mu, log_var, Z = self.forward(Xt, rng_train)
            recon_loss = torch.mean((X_hat - Xt) ** 2)
            kl_loss = self.kl_divergence(mu, log_var)
            total_loss = recon_loss + beta * kl_loss
            history['loss'].append(float(total_loss))
            history['recon'].append(float(recon_loss))
            history['kl'].append(float(kl_loss))
            for p in params:
                p.grad = None
            total_loss.backward()
            with torch.no_grad():
                for p in params:
                    g = p.grad
                    norm = float(torch.linalg.norm(g))
                    if norm > 5.0:  # the NumPy lane's per-gradient clip
                        g = g * (5.0 / norm)
                    p -= lr * g
            if verbose and (step + 1) % 1000 == 0:
                print(f"  step {step + 1}/{n_steps}, loss = {history['loss'][-1]:.4f}")
        for p in params:
            p.requires_grad_(False)
        return history


In [ ]:
# exports: final_loss, final_kl, mu_head, recon_head
_rng_eq = np.random.default_rng(1700)
_t_eq = _rng_eq.uniform(0, 2 * np.pi, 60)
_r_eq = 1.0 + 0.3 * _rng_eq.normal(size=60)
_Zlat_eq = np.column_stack([_r_eq * np.cos(_t_eq), _r_eq * np.sin(_t_eq)])
_A_eq = _rng_eq.normal(size=(2, 6))
_X_eq = _Zlat_eq @ _A_eq + 0.1 * _rng_eq.normal(size=(60, 6))
_X_eq = _X_eq - _X_eq.mean(axis=0)

_vae_eq = VAE(6, d_hidden=12, d_latent=2, random_state=9)
_hist_eq = _vae_eq.fit(_X_eq, lr=0.01, n_steps=300, beta=0.1, verbose=False)
_rng_eval_eq = np.random.default_rng(999)
_Xh_eq, _mu_eq, _lv_eq, _Zv_eq = _vae_eq.forward(_X_eq, _rng_eval_eq)
final_loss = _hist_eq['loss'][-1]
final_kl = _hist_eq['kl'][-1]
mu_head = _mu_eq[:5].detach().numpy()
recon_head = _Xh_eq[:5].detach().numpy()
print(f"final ELBO loss: {final_loss:.6f}  (KL term: {final_kl:.6f})")


In [ ]:
_zeros_eq = torch.zeros((4, 2), dtype=torch.float64)
_kl0_eq = float(VAE.kl_divergence(_zeros_eq, _zeros_eq))
assert _kl0_eq == 0.0, "KL(N(0,I) || N(0,I)) is exactly zero"
assert final_kl > 0.0, "a trained posterior is not the prior"
# The reparameterised sample is a deterministic function of the NumPy seed.
_h1_eq = _vae_eq.forward(_X_eq, np.random.default_rng(1))[0]
_h2_eq = _vae_eq.forward(_X_eq, np.random.default_rng(1))[0]
assert float(torch.max(torch.abs(_h1_eq - _h2_eq))) == 0.0, "same seed, same sample"
assert _hist_eq['loss'][-1] < _hist_eq['loss'][0], "the ELBO objective improved over training"
